# Solución — Ejercicio 01: agregar experiment tracking

Solución de referencia de
[`../exercises/ejercicio-01.md`](../exercises/ejercicio-01.md).

**No la publiques antes del taller.** Y si estás resolviendo el ejercicio,
inténtalo primero: el valor está en pelearse con la UI, no en leer esto.

Prerrequisitos: `make data` y `make mlflow`.

In [ ]:
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import mlflow
import pandas as pd
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error

from taxi import config
from taxi.features import contract as fc
from taxi.models import train

DIR_SALIDA = Path(tempfile.mkdtemp(prefix="solucion-01-"))
print("artefactos temporales en:", DIR_SALIDA)

## Parte 1 — Código base (igual que en el enunciado)

In [ ]:
df_train = train.cargar_train()
df_valid = train.cargar_valid()
y_valid = df_valid[fc.TARGET_REGRESION].to_numpy(dtype=float)

n_estimators = 25
max_depth = 10
random_state = config.SEMILLA

modelo = train.pipeline_random_forest(
    n_estimators=n_estimators,
    max_depth=max_depth,
    random_state=random_state,
)
train.ajustar(modelo, df_train, df_valid)
y_pred = modelo.predict(df_valid)

rmse = float(root_mean_squared_error(y_valid, y_pred))
mae = float(mean_absolute_error(y_valid, y_pred))
r2 = float(r2_score(y_valid, y_pred))
print(f"RMSE={rmse:.4f} MAE={mae:.4f} R2={r2:.4f}")

In [ ]:
df_predicciones = pd.DataFrame({"y_true": y_valid, "y_pred": y_pred, "error": y_valid - y_pred})
ruta_csv = DIR_SALIDA / "predictions.csv"
df_predicciones.to_csv(ruta_csv, index=False)

fig, eje = plt.subplots(figsize=(8, 4))
eje.hist(y_valid - y_pred, bins=50, edgecolor="black", alpha=0.7)
eje.set_title("Distribucion de residuales (y_true - y_pred)")
eje.set_xlabel("residual (minutos)")
eje.axvline(x=0, color="red", linestyle="--", alpha=0.6)
fig.tight_layout()
ruta_png = DIR_SALIDA / "residuals.png"
fig.savefig(ruta_png, dpi=150)
plt.close(fig)
print(ruta_csv.name, ruta_png.name)

## TODO 1 — Conexión

`config.MLFLOW_TRACKING_URI` en lugar de la URL literal: si el puerto cambia,
cambia en un solo sitio del repositorio.

In [ ]:
mlflow.set_tracking_uri(config.MLFLOW_TRACKING_URI)
mlflow.set_experiment("nyc-taxi-ejercicio-01")
print("tracking URI:", mlflow.get_tracking_uri())

## TODO 2 a 5 — El run completo

Tres decisiones que valen más que el código:

1. El tag `dataset` nombra **particiones concretas**. `nyc_taxi` no distingue dos
   runs entrenados con meses distintos.
2. `filas_train` y `filas_valid` van como params: son parte de "con qué se
   entrenó", y detectan que alguien cambió el muestreo.
3. `holdout_evaluado=no` deja explícito que este run **no** miró el conjunto de
   test. Cuando el gate de S06 lo mire, el tag cambiará, y en la UI se verá
   exactamente qué runs lo consultaron.

In [ ]:
with mlflow.start_run(run_name="rf-ejercicio-01") as run:
    # TODO 2: tags
    mlflow.set_tag("problem_type", "regression")
    mlflow.set_tag("model_family", "random_forest")
    mlflow.set_tag("dataset", ",".join(p.etiqueta for p in config.PARTICIONES_TRAIN))
    mlflow.set_tag("holdout_evaluado", "no")

    # TODO 3: params
    mlflow.log_param("n_estimators", n_estimators)
    mlflow.log_param("max_depth", max_depth)
    mlflow.log_param("random_state", random_state)
    mlflow.log_param("filas_train", len(df_train))
    mlflow.log_param("filas_valid", len(df_valid))

    # TODO 4: metricas
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2", r2)

    # TODO 5: artefactos
    mlflow.log_artifact(str(ruta_csv))
    mlflow.log_artifact(str(ruta_png))

    run_id = run.info.run_id

print("run_id:", run_id)

## Verificación — se cuenta, no se opina

In [ ]:
cliente = mlflow.MlflowClient()
datos = cliente.get_run(run_id).data
artefactos = [a.path for a in cliente.list_artifacts(run_id)]
propios = {k: v for k, v in datos.tags.items() if not k.startswith("mlflow.")}

print(f"tags       : {len(propios)} -> {sorted(propios)}")
print(f"parametros : {len(datos.params)} -> {sorted(datos.params)}")
print(f"metricas   : {len(datos.metrics)} -> {sorted(datos.metrics)}")
print(f"artefactos : {len(artefactos)} -> {artefactos}")

## Bonus 3 — Qué registra `autolog()` y qué no

Con `autolog()` aparecen los ~30 params del estimador y varias métricas de
entrenamiento sin escribir una línea. Lo que **no** aparece es todo lo que no
está en la llamada a `fit`:

| Autolog sí | Autolog no |
|---|---|
| hiperparámetros del estimador | qué particiones de datos usaste |
| métricas de entrenamiento | tu métrica de negocio |
| el modelo con firma inferida | métricas por subgrupo |
| algunos artifacts del framework | `holdout_evaluado`, `features`, `semilla` |

Conclusión práctica: `autolog()` para params y modelo, logging manual para los
**tags de datos** y las **métricas de decisión**. Y desactívalo al terminar
(`mlflow.sklearn.autolog(disable=True)`): si queda activo, cada `fit` posterior
crea un run y ensucia el experimento.

In [ ]:
mlflow.sklearn.autolog()
with mlflow.start_run(run_name="rf-autolog-bonus") as run_auto:
    mlflow.set_tag("dataset", ",".join(p.etiqueta for p in config.PARTICIONES_TRAIN))
    modelo_auto = train.pipeline_random_forest(n_estimators=25, max_depth=10)
    train.ajustar(modelo_auto, df_train, df_valid)
    run_id_auto = run_auto.info.run_id
mlflow.sklearn.autolog(disable=True)

auto = mlflow.get_run(run_id_auto).data
print("params de autolog:", len(auto.params))
print("metricas de autolog:", sorted(auto.metrics))
print("tags propios:", [k for k in auto.tags if not k.startswith("mlflow.")])

## Bonus 4 — Loguear el modelo con `signature` e `input_example`

Dos cosas que solo se descubren haciéndolo:

- `train._ejemplo_de_entrada` **ensancha** los tipos (`int16` → `int64`). Con el
  dataframe tal cual, la firma quedaría en `int32` y MLflow rechazaría cualquier
  petición con `int64`: `Can not safely convert int64 to int32`.
- Sin `skops_trusted_types`, `log_model` **falla**: en MLflow 3 el
  `serialization_format` por defecto es `skops`, que solo reconstruye tipos de una
  allowlist. `ADiccionarios` no está en ella porque es una clase del curso.

In [ ]:
from mlflow.models import infer_signature

ejemplo = train._ejemplo_de_entrada(df_valid)
firma = infer_signature(ejemplo, modelo.predict(ejemplo))

with mlflow.start_run(run_id=run_id):
    info = mlflow.sklearn.log_model(
        sk_model=modelo,
        name="modelo",
        signature=firma,
        input_example=ejemplo,
        skops_trusted_types=["taxi.models.train.ADiccionarios"],
    )

print("model_uri:", info.model_uri)
print(firma)

In [ ]:
cargado = mlflow.pyfunc.load_model(info.model_uri)
malo = ejemplo.copy()
malo["hora_pickup"] = malo["hora_pickup"].astype("float64")
try:
    cargado.predict(malo)
    print("no fallo: revisa la firma")
except Exception as error:
    texto = " ".join(str(error).splitlines())
    corte = texto.find("Error:")
    print(type(error).__name__, "|", texto[corte:][:160])

## Cierre para el instructor

Los errores que aparecen todos los años, en orden de frecuencia:

| Síntoma | Causa |
|---|---|
| "no veo el run" | `set_experiment` con otro nombre, o el server en otro puerto |
| 2 artefactos y aparecen 0 | se llamó a `log_artifact` **fuera** del `with` |
| "los tags no salen" | se usó `log_param` en lugar de `set_tag` |
| "el rmse sale rarísimo" | se comparó contra `y_train` en lugar de `y_valid` |
| `HTTP 403` | AirPlay en el puerto por defecto de MLflow, en macOS |